In [ ]:
# Simple Moirai Inference and Visualization
# Load model, run inference, and visualize results

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gluonts.dataset.pandas import PandasDataset
from gluonts.dataset.split import split
from huggingface_hub import hf_hub_download
import warnings
warnings.filterwarnings('ignore')

from uni2ts.model.moirai import MoiraiForecast, MoiraiModule

In [ ]:
# Configuration
MODEL = "moirai"  # or "moirai-moe"
SIZE = "large"    # small, base, large
CTX = 128          # Context length
PDT = 16           # Prediction length
BSZ = 32          # Batch size
GPU = 0           # GPU device
PSZ = "auto"

# Data configuration
CSV_PATH = "/home/sa53869/time-series/moirai/time-moe-eval/synthetic_sinusoidal.csv"
COLUMN = 1        # Column to analyze (0-indexed)
TEST_LENGTH = 800 # Test set length

# Set GPU
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU)

print(f"Configuration:")
print(f"  Model: {MODEL}-{SIZE}")
print(f"  Context Length: {CTX}")
print(f"  Prediction Length: {PDT}")
print(f"  Test Length: {TEST_LENGTH}")
print(f"  Using GPU: {GPU}")

In [ ]:
# Load and prepare data
print("Loading data...")
df = pd.read_csv(CSV_PATH, index_col=0, parse_dates=True)
dataset_name = os.path.splitext(os.path.basename(CSV_PATH))[0]

# Select column
available_columns = df.columns.tolist()
selected_column = available_columns[COLUMN]
df_selected = df[[selected_column]].copy()

print(f"Dataset: {dataset_name}")
print(f"Selected column: {selected_column}")
print(f"Data shape: {df_selected.shape}")
print(f"Data preview:")
print(df_selected.head())

# Create GluonTS dataset
ds = PandasDataset(dict(df_selected))
train, test_template = split(ds, offset=-TEST_LENGTH)

# Generate test instances
test_data = test_template.generate_instances(
    prediction_length=PDT,
    windows=TEST_LENGTH // PDT,
    distance=PDT,
)

print(f"Number of test windows: {TEST_LENGTH // PDT}")

In [ ]:
# Load Moirai model
print("Loading Moirai model...")
base_module = MoiraiModule.from_pretrained(f"Salesforce/moirai-1.0-R-{SIZE}")

# Create model with specific configuration
model = MoiraiForecast(
    module=base_module,
    prediction_length=PDT,
    context_length=CTX,
    patch_size=PSZ,
    num_samples=100,  # Number of samples for probabilistic forecasting
    target_dim=1,
    feat_dynamic_real_dim=ds.num_feat_dynamic_real,
    past_feat_dynamic_real_dim=ds.num_past_feat_dynamic_real,
)

# Create predictor
predictor = model.create_predictor(batch_size=BSZ)
print("Model loaded successfully!")

In [ ]:
# Run inference on test data
print("Running inference...")
input_data = list(test_data.input)
label_data = list(test_data.label)

# Run predictions
forecasts = list(predictor.predict(input_data))

print(f"Generated {len(forecasts)} forecasts")

# Prepare data for visualization
sample_results = []
full_data_values = df_selected[selected_column].values

for i, (input_item, label_item, forecast) in enumerate(zip(input_data, label_data, forecasts)):
    # Get context data
    context = input_item['target']
    
    # Get ground truth
    ground_truth = label_item['target'][:PDT]
    
    # Get prediction (mean of samples)
    prediction = np.mean(forecast.samples, axis=0)
    
    # Store results
    sample_results.append({
        'window_id': i,
        'context': context,
        'ground_truth': ground_truth,
        'prediction': prediction,
        'mae': np.mean(np.abs(prediction - ground_truth))
    })

print(f"Processed {len(sample_results)} samples")

In [ ]:
# Visualization function
def plot_forecast_sample(context, ground_truth, prediction, window_id, mae_score):
    """Plot a single forecast sample with context, ground truth, and prediction"""
    
    # Ensure we only use the actual context length that the model sees
    # Limit context to CTX length if longer
    if len(context) > CTX:
        actual_context = context[-CTX:]  # Take only the last CTX points
    else:
        actual_context = context
    
    context_length = len(actual_context)
    
    # Create time indices - context goes from -context_length to 0, forecast from 0 to PDT
    context_indices = np.arange(-context_length, 0)
    forecast_indices = np.arange(0, len(ground_truth))
    
    plt.figure(figsize=(14, 7))
    
    # Plot context (historical data that model sees)
    plt.plot(context_indices, actual_context, 'b-', linewidth=2.5, 
             label=f'Context ({context_length} steps)', alpha=0.8)
    
    # Plot ground truth forecast
    plt.plot(forecast_indices, ground_truth, 'g-', linewidth=3, 
             label='Ground Truth', marker='o', markersize=7, markerfacecolor='lightgreen')
    
    # Plot model prediction
    plt.plot(forecast_indices, prediction, 'r--', linewidth=2.5, 
             label='Model Prediction', marker='s', markersize=6, markerfacecolor='lightcoral')
    
    # Add vertical line to clearly separate context from forecast
    plt.axvline(x=0, color='black', linestyle='-', linewidth=2, alpha=0.8, label='Forecast Start')
    
    # Add shaded regions to distinguish context from forecast
    plt.axvspan(-context_length, 0, alpha=0.1, color='blue', label='Context Region')
    plt.axvspan(0, len(ground_truth), alpha=0.1, color='green', label='Forecast Region')
    
    # Formatting
    plt.title(f'Sample {window_id + 1}: Moirai Inference Results\n'
              f'Context: {context_length} steps → Forecast: {len(ground_truth)} steps | MAE: {mae_score:.4f}', 
              fontsize=14, fontweight='bold')
    plt.xlabel('Time Steps (relative to forecast start)', fontsize=12)
    plt.ylabel('Value', fontsize=12)
    plt.legend(loc='upper left', fontsize=10)
    plt.grid(True, alpha=0.3)
    
    # Add text box with key information
    stats_text = f'Model: {MODEL}-{SIZE}\nContext Length: {context_length}\nForecast Length: {len(ground_truth)}\nMAE Score: {mae_score:.4f}'
    plt.text(0.98, 0.98, stats_text, transform=plt.gca().transAxes, 
             bbox=dict(boxstyle='round,pad=0.5', facecolor='wheat', alpha=0.9),
             verticalalignment='top', horizontalalignment='right', fontsize=10)
    
    plt.tight_layout()
    plt.show()

print("Updated visualization function defined")

In [ ]:
# Select 4 random samples and create a single figure with subfigures (2x2 grid)
print(f"Available samples: {len(sample_results)}")

# Generate random indices for sample selection
np.random.seed(42)  # For reproducible results
max_samples = len(sample_results)

if max_samples == 0:
    print("Error: No sample results available!")
else:
    # Select up to 4 samples
    n_samples = min(4, max_samples)
    selected_indices = np.random.choice(max_samples, n_samples, replace=False)
    print(f"Selected indices: {selected_indices}")

    # Create the figure
    fig, axes = plt.subplots(2, 2, figsize=(20, 12))
    fig.suptitle(f'Moirai Forecasting Results: {n_samples} Random Samples\n'
                 f'Model: {MODEL}-{SIZE} | Context: {CTX} | Prediction: {PDT}', 
                 fontsize=16, fontweight='bold')

    axes = axes.flatten()  # Flatten for easy indexing

    print(f"Plotting {n_samples} samples in a single figure:")

    # Plot each selected sample in its subplot
    for i, idx in enumerate(selected_indices):
        sample = sample_results[idx]
        ax = axes[i]
        
        # Get data
        context = sample['context']
        ground_truth = sample['ground_truth']
        prediction = sample['prediction']
        
        # Ensure we only use the model's context length
        if len(context) > CTX:
            context = context[-CTX:]
        
        # Create time indices
        context_length = len(context)
        context_indices = np.arange(-context_length, 0)
        forecast_indices = np.arange(0, len(ground_truth))
        
        # Plot context
        ax.plot(context_indices, context, 'b-', linewidth=2, label='Context', alpha=0.8)
        
        # Plot ground truth
        ax.plot(forecast_indices, ground_truth, 'g-', linewidth=3, 
                label='Ground Truth', marker='o', markersize=5)
        
        # Plot prediction
        ax.plot(forecast_indices, prediction, 'r--', linewidth=2, 
                label='Prediction', marker='s', markersize=5)
        
        # Add vertical line at forecast start
        ax.axvline(x=0, color='black', linestyle=':', alpha=0.7, linewidth=1)
        
        # Formatting
        ax.set_title(f'Sample {i+1} (Window {sample["window_id"] + 1})\n'
                    f'MAE: {sample["mae"]:.4f}', fontsize=12)
        ax.set_xlabel('Time Steps')
        ax.set_ylabel('Value')
        ax.grid(True, alpha=0.3)
        
        # Add legend only to the first subplot to avoid clutter
        if i == 0:
            ax.legend(loc='upper left', fontsize=10)
        
        # Add text box with key info
        stats_text = f'Ctx: {context_length}\nPred: {len(ground_truth)}\nMAE: {sample["mae"]:.3f}'
        ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, 
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
                verticalalignment='top', fontsize=9)

    # Hide empty subplots if we have fewer than 4 samples
    for i in range(n_samples, 4):
        axes[i].set_visible(False)

    plt.tight_layout()
    plt.show()

    print(f"\nAll {n_samples} samples plotted in a single figure!")
    
    # Print summary of selected samples
    print(f"\nSelected samples summary:")
    for i, idx in enumerate(selected_indices):
        sample = sample_results[idx]
        print(f"  Sample {i+1}: Window {sample['window_id'] + 1}, MAE = {sample['mae']:.4f}")

In [ ]:
# Overall performance summary
print("\n" + "="*60)
print("OVERALL PERFORMANCE SUMMARY")
print("="*60)

# Calculate statistics
mae_scores = [sample['mae'] for sample in sample_results]
mean_mae = np.mean(mae_scores)
std_mae = np.std(mae_scores)
min_mae = np.min(mae_scores)
max_mae = np.max(mae_scores)

print(f"Dataset: {dataset_name}")
print(f"Column: {selected_column}")
print(f"Model: {MODEL}-{SIZE}")
print(f"Context Length: {CTX}")
print(f"Prediction Length: {PDT}")
print(f"Number of test windows: {len(sample_results)}")
print(f"\nMAE Statistics:")
print(f"  Mean MAE: {mean_mae:.4f}")
print(f"  Std MAE:  {std_mae:.4f}")
print(f"  Min MAE:  {min_mae:.4f}")
print(f"  Max MAE:  {max_mae:.4f}")

# Find best and worst performing samples
best_idx = np.argmin(mae_scores)
worst_idx = np.argmax(mae_scores)

print(f"\nBest Performance:")
print(f"  Window {sample_results[best_idx]['window_id'] + 1}: MAE = {sample_results[best_idx]['mae']:.4f}")
print(f"Worst Performance:")
print(f"  Window {sample_results[worst_idx]['window_id'] + 1}: MAE = {sample_results[worst_idx]['mae']:.4f}")

print(f"\nSelected samples for visualization:")
for i, idx in enumerate(selected_indices):
    sample = sample_results[idx]
    print(f"  Sample {i+1}: Window {sample['window_id'] + 1}, MAE = {sample['mae']:.4f}")

In [ ]:
# Optional: Plot distribution of MAE scores
plt.figure(figsize=(10, 6))
plt.hist(mae_scores, bins=30, alpha=0.7, color='skyblue', edgecolor='black')
plt.axvline(mean_mae, color='red', linestyle='--', linewidth=2, label=f'Mean MAE: {mean_mae:.4f}')
plt.xlabel('MAE Score')
plt.ylabel('Frequency')
plt.title(f'Distribution of MAE Scores\n{MODEL}-{SIZE} on {dataset_name} ({selected_column})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Analysis complete!")

In [ ]:
# Context Reduction Experiment: Drop Every Alternate Sample
print("="*60)
print("CONTEXT REDUCTION EXPERIMENT")
print("="*60)
print("Comparing full context vs. reduced context (every alternate sample)")

# Create a model with reduced context length for fair comparison
reduced_ctx_length = CTX // 2
print(f"Original context length: {CTX}")
print(f"Reduced context length: {reduced_ctx_length}")

# Create model with reduced context length
model_reduced = MoiraiForecast(
    module=base_module,
    prediction_length=PDT,
    context_length=reduced_ctx_length,
    patch_size=16,
    num_samples=100,
    target_dim=1,
    feat_dynamic_real_dim=ds.num_feat_dynamic_real,
    past_feat_dynamic_real_dim=ds.num_past_feat_dynamic_real,
)

predictor_reduced = model_reduced.create_predictor(batch_size=BSZ)
print("Reduced context model created successfully!")

# Process samples with reduced context
print(f"\nProcessing {len(input_data)} samples with reduced context...")
reduced_sample_results = []

for i, (input_item, label_item) in enumerate(zip(input_data, label_data)):
    # Get original context
    original_context = input_item['target']
    
    # Drop every alternate sample (subsample by factor of 2)
    if len(original_context) > reduced_ctx_length:
        # Take every 2nd sample to get approximately half the data
        step = max(1, len(original_context) // reduced_ctx_length)
        reduced_context = original_context[::step][:reduced_ctx_length]
    else:
        reduced_context = original_context
    
    # Create input with reduced context
    reduced_input = {
        'target': reduced_context,
        'start': input_item['start'],
        'item_id': input_item.get('item_id', 0)
    }
    
    # Get prediction with reduced context
    forecast_reduced = next(iter(predictor_reduced.predict([reduced_input])))
    prediction_reduced = np.mean(forecast_reduced.samples, axis=0)
    
    # Get ground truth
    ground_truth = label_item['target'][:PDT]
    
    # Calculate MAE
    mae_reduced = np.mean(np.abs(prediction_reduced - ground_truth))
    
    # Store results
    reduced_sample_results.append({
        'window_id': i,
        'original_context': original_context,
        'reduced_context': reduced_context,
        'ground_truth': ground_truth,
        'prediction_reduced': prediction_reduced,
        'prediction_full': sample_results[i]['prediction'],
        'mae_reduced': mae_reduced,
        'mae_full': sample_results[i]['mae']
    })
    
    if (i + 1) % 20 == 0:
        print(f"Processed {i + 1}/{len(input_data)} samples")

print(f"Context reduction processing complete!")

# Calculate performance statistics
mae_full_list = [r['mae_full'] for r in reduced_sample_results]
mae_reduced_list = [r['mae_reduced'] for r in reduced_sample_results]

mean_mae_full = np.mean(mae_full_list)
mean_mae_reduced = np.mean(mae_reduced_list)
performance_drop = ((mean_mae_reduced - mean_mae_full) / mean_mae_full) * 100

print(f"\nPerformance Comparison:")
print(f"Full Context (CTX={CTX}):     Mean MAE = {mean_mae_full:.4f}")
print(f"Reduced Context (CTX={reduced_ctx_length}): Mean MAE = {mean_mae_reduced:.4f}")
print(f"Performance Drop: {performance_drop:+.2f}%")